# LeetCode 292 — Nim Game bằng Python

**Link đề:** https://leetcode.com/problems/nim-game/
**Độ khó:** Easy · **Chủ đề:** Math, Game Theory (lý thuyết trò chơi), Brainteaser

```python
class Solution:
    def canWinNim(self, n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu cách mô hình hóa một **trò chơi hai người chơi tối ưu** thành bài toán "thế thắng / thế thua".
2. Viết được lời giải **đệ quy** trực giác, thấy vì sao nó bùng nổ thời gian.
3. Dùng **memoization** (`functools.lru_cache`) và **bảng DP** để tính nhanh hơn.
4. Từ bảng kết quả, **phát hiện quy luật** `n % 4 != 0` và **chứng minh** được nó.
5. Biết khi nào một bài "trò chơi" có công thức O(1), và cách kiểm chứng công thức bằng brute force.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — ai lấy viên đá cuối cùng thì thắng

Trên bàn có một đống `n` viên đá. Hai người chơi (bạn và một người bạn) lần lượt đi, **bạn đi trước**.

- Mỗi lượt, người chơi phải lấy **1, 2 hoặc 3** viên đá.
- Người lấy được **viên đá cuối cùng** là người **thắng**.

Giả sử **cả hai đều chơi tối ưu** (luôn chọn nước đi tốt nhất cho mình). Trả về `True` nếu bạn chắc chắn thắng, ngược lại `False`.

| Input `n` | Output | Giải thích |
|---:|:---:|---|
| `4` | `False` | Bạn lấy 1/2/3 thì bạn kia còn 3/2/1 viên → bạn kia lấy hết và thắng |
| `1` | `True` | Bạn lấy luôn viên duy nhất |
| `2` | `True` | Bạn lấy cả 2 viên |

**Constraints:** `1 <= n <= 2^31 - 1`.

**Gợi ý của đề:** Nếu có 5 viên đá, bạn có cách nào luôn thắng không?

Câu hỏi đặt ra trước khi code:
- "Chơi tối ưu" nghĩa là gì khi viết thành code?
- Có cần mô phỏng mọi ván đấu không? Với `n` tới hơn 2 tỉ thì sao?

In [1]:
# Ví dụ nhỏ: liệt kê mọi nước đi đầu tiên khi n = 4
n = 4
for take in (1, 2, 3):
    remain = n - take
    print(f"Bạn lấy {take} -> còn {remain} viên cho bạn kia (bạn kia lấy hết {remain} và thắng)")

Bạn lấy 1 -> còn 3 viên cho bạn kia (bạn kia lấy hết 3 và thắng)
Bạn lấy 2 -> còn 2 viên cho bạn kia (bạn kia lấy hết 2 và thắng)
Bạn lấy 3 -> còn 1 viên cho bạn kia (bạn kia lấy hết 1 và thắng)


## 2. Nền tảng: thế thắng (winning position) và thế thua (losing position)

Trong các trò chơi hai người, lần lượt, không may rủi, thông tin đầy đủ, ta gọi **trạng thái** là "số đá còn lại khi tới lượt một người nào đó".

- **Thế thắng (W)**: người đang đến lượt có **ít nhất một** nước đi đưa đối thủ vào **thế thua**.
- **Thế thua (L)**: **mọi** nước đi đều đưa đối thủ vào **thế thắng** (hoặc không còn nước đi).

Đây chính là định nghĩa của "chơi tối ưu": người chơi luôn tìm nước đi khiến đối thủ rơi vào thế thua, nếu có.

Với Nim Game của bài:
- `n = 0` khi đến lượt bạn: viên cuối đã bị đối thủ lấy → bạn **đã thua** → `0` là thế **L**.
- `n = 1, 2, 3`: lấy hết là thắng → thế **W**.
- `n = 4`: lấy 1/2/3 đều để lại 3/2/1 (thế W cho đối thủ) → thế **L**.

Ta tính bằng tay vài giá trị đầu:

| n | Các nước đi → trạng thái đối thủ | Có nước đưa đối thủ về L? | Kết luận |
|---:|---|:---:|:---:|
| 0 | (không còn đá) | — | L |
| 1 | 0(L) | có | W |
| 2 | 1(W), 0(L) | có | W |
| 3 | 2(W), 1(W), 0(L) | có | W |
| 4 | 3(W), 2(W), 1(W) | không | **L** |
| 5 | 4(L), 3(W), 2(W) | có (lấy 1) | W |
| 8 | 7(W), 6(W), 5(W) | không | **L** |

### 2.1. `any()` và `all()` — công cụ Python để viết "có ít nhất một" / "mọi"

- `any(iterable)` trả `True` nếu **ít nhất một** phần tử là `True`.
- `all(iterable)` trả `True` nếu **mọi** phần tử là `True`.

Hai hàm này khớp chính xác với định nghĩa W/L ở trên: "thế thắng nếu **any** nước đi dẫn đối thủ về thế thua".

In [2]:
print(any([False, True, False]))   # True: có ít nhất một True
print(any([False, False]))         # False
print(all([True, True]))           # True: mọi phần tử đều True
print(all([True, False]))          # False
print(any([]))                     # False: list rỗng không có phần tử True nào

True
False
True
False
False


### 2.2. Generator expression bên trong `any()`

Thay vì tạo list rồi mới gọi `any`, ta viết thẳng:

```python
any(điều_kiện(k) for k in (1, 2, 3))
```

Đây là **generator expression**: sinh giá trị từng cái một, và `any` **dừng sớm** ngay khi gặp `True` (short-circuit).

In [3]:
def noisy_check(k):
    print("  đang kiểm tra k =", k)
    return k == 2

print("Kết quả:", any(noisy_check(k) for k in (1, 2, 3)))  # dừng ở k = 2, không kiểm tra k = 3

  đang kiểm tra k = 1
  đang kiểm tra k = 2
Kết quả: True


### 2.3. Toán tử `%` (chia lấy dư)

`a % b` là phần dư khi chia `a` cho `b`. Với `b = 4`, phần dư chỉ có thể là `0, 1, 2, 3`.

`n % 4 == 0` nghĩa là `n` **chia hết cho 4** (là bội của 4).

In [4]:
for n in range(0, 10):
    print(n, "% 4 =", n % 4, "| chia hết cho 4?" , n % 4 == 0)

0 % 4 = 0 | chia hết cho 4? True
1 % 4 = 1 | chia hết cho 4? False
2 % 4 = 2 | chia hết cho 4? False
3 % 4 = 3 | chia hết cho 4? False
4 % 4 = 0 | chia hết cho 4? True
5 % 4 = 1 | chia hết cho 4? False
6 % 4 = 2 | chia hết cho 4? False
7 % 4 = 3 | chia hết cho 4? False
8 % 4 = 0 | chia hết cho 4? True
9 % 4 = 1 | chia hết cho 4? False


## 3. Cách trực giác — đệ quy mô phỏng "chơi tối ưu"

Dịch thẳng định nghĩa W/L thành code:

```text
win(n) = True  nếu tồn tại k ∈ {1,2,3}, k <= n, mà win(n - k) == False
win(0) = False
```

Đọc là: "Tôi thắng ở `n` nếu tôi có thể lấy `k` viên sao cho đối thủ (đứng ở `n - k`) **không** thắng."

In [5]:
def can_win_recursive(n: int) -> bool:
    if n == 0:
        return False          # đến lượt mình mà không còn đá: đối thủ đã lấy viên cuối
    return any(not can_win_recursive(n - k) for k in (1, 2, 3) if k <= n)

for n in range(1, 10):
    print(n, can_win_recursive(n))

1 True
2 True
3 True
4 False
5 True
6 True
7 True
8 False
9 True


### 3.1. Đếm số lần gọi hàm — thấy chi phí của đệ quy thuần

Mỗi lời gọi `win(n)` có thể gọi tiếp tới 3 lời gọi con. Không nhớ kết quả cũ nên cùng một `n` bị tính lại rất nhiều lần. Ta đếm bằng một biến toàn cục `calls`.

(Ta dùng bản **không dừng sớm** để thấy trường hợp xấu: duyệt cả 3 nước đi.)

In [6]:
calls = 0

def can_win_count(n: int) -> bool:
    global calls
    calls += 1
    if n == 0:
        return False
    results = [not can_win_count(n - k) for k in (1, 2, 3) if k <= n]   # list: tính hết, không dừng sớm
    return any(results)

for n in [5, 10, 15, 20]:
    calls = 0
    can_win_count(n)
    print(f"n = {n:>2}: số lần gọi = {calls}")

n =  5: số lần gọi = 28
n = 10: số lần gọi = 600
n = 15: số lần gọi = 12640
n = 20: số lần gọi = 266079


## 4. Nút thắt của cách trực giác

- Số lần gọi tăng **theo cấp số mũ** (gần như nhân ~1.8 mỗi khi `n` tăng 1). Với `n = 40` đã là hàng tỉ lần gọi.
- Ngay cả khi nhớ kết quả (memo), ta vẫn phải tính **mọi** `n` từ 0 tới `n`: với `n = 2^31 - 1` là hơn 2 tỉ bước → quá chậm và tốn bộ nhớ.
- Đệ quy sâu `n` tầng còn gây **RecursionError** (Python mặc định giới hạn khoảng 1000 tầng).

Vậy ta cần: (1) loại bỏ tính lặp, (2) cuối cùng tìm **công thức** không phụ thuộc vào việc duyệt tới `n`.

In [7]:
import sys
print("Giới hạn đệ quy mặc định:", sys.getrecursionlimit())

Giới hạn đệ quy mặc định: 1000


## 5. Bước cải tiến 1 — memoization với `functools.lru_cache`

**Memoization**: lưu kết quả của lời gọi hàm theo tham số; lần sau gọi lại cùng tham số thì trả ngay kết quả đã lưu.

Trong Python, decorator `@lru_cache(maxsize=None)` (hoặc `@cache` từ Python 3.9) làm việc đó tự động.

- **Decorator** là cú pháp `@ten` đặt trên `def`, "bọc" hàm bằng một hàm khác để thêm hành vi.

In [8]:
from functools import lru_cache

@lru_cache(maxsize=None)
def can_win_memo(n: int) -> bool:
    if n == 0:
        return False
    return any(not can_win_memo(n - k) for k in (1, 2, 3) if k <= n)

print([can_win_memo(n) for n in range(1, 13)])
print(can_win_memo.cache_info())   # hits = số lần lấy từ bộ nhớ đệm, misses = số lần tính mới

[True, True, True, False, True, True, True, False, True, True, True, False]
CacheInfo(hits=26, misses=13, maxsize=None, currsize=13)


## 6. Bước cải tiến 2 — bảng DP từ dưới lên (bottom-up)

Thay đệ quy bằng vòng lặp: `dp[i]` = "người đến lượt với `i` viên có thắng không?".

```text
dp[0] = False
dp[i] = (not dp[i-1]) or (not dp[i-2]) or (not dp[i-3])   (chỉ xét chỉ số >= 0)
```

Cách này không có đệ quy sâu, chạy O(n) thời gian và O(n) bộ nhớ. Ta dùng nó để **in bảng** và quan sát.

In [9]:
def nim_table(n: int) -> list[bool]:
    dp = [False] * (n + 1)             # dp[0] = False
    for i in range(1, n + 1):
        dp[i] = any(not dp[i - k] for k in (1, 2, 3) if i - k >= 0)
    return dp

table = nim_table(16)
for i, w in enumerate(table):
    print(f"n = {i:>2} -> {'W (thắng)' if w else 'L (thua)'}")

n =  0 -> L (thua)
n =  1 -> W (thắng)
n =  2 -> W (thắng)
n =  3 -> W (thắng)
n =  4 -> L (thua)
n =  5 -> W (thắng)
n =  6 -> W (thắng)
n =  7 -> W (thắng)
n =  8 -> L (thua)
n =  9 -> W (thắng)
n = 10 -> W (thắng)
n = 11 -> W (thắng)
n = 12 -> L (thua)
n = 13 -> W (thắng)
n = 14 -> W (thắng)
n = 15 -> W (thắng)
n = 16 -> L (thua)


## 7. Insight — quan sát bảng và phát hiện quy luật

Nhìn bảng trên: các thế **L** là `0, 4, 8, 12, 16` — đúng là **bội của 4**.

Giả thuyết: **Bạn thua khi và chỉ khi `n % 4 == 0`.**

Vì sao? Ý tưởng "cặp lượt đi có tổng 4":
- Nếu đối thủ lấy `k` viên (`k ∈ {1,2,3}`), bạn đáp lại `4 - k` viên (cũng ∈ {1,2,3}). Sau một cặp lượt, số đá giảm **đúng 4**.
- Nếu `n` **không** chia hết cho 4: ở lượt đầu bạn lấy `n % 4` viên → còn bội của 4 cho đối thủ. Từ đó bạn luôn đáp `4 - k`, giữ số đá là bội của 4 khi tới lượt đối thủ. Cuối cùng đối thủ đứng ở `0`… nghĩa là bạn vừa lấy viên cuối.
- Nếu `n` **chia hết** cho 4: đối thủ dùng chính chiến lược đó với bạn → bạn thua.

Trả lời gợi ý của đề với `n = 5`: bạn lấy **1** viên, còn 4 viên cho đối thủ — thế thua của họ.

In [10]:
# Kiểm tra giả thuyết bằng bảng DP cho n = 0..200
dp = nim_table(200)
mismatch = [i for i in range(201) if dp[i] != (i % 4 != 0)]
print("Các n mà giả thuyết sai:", mismatch)   # mong đợi: []

Các n mà giả thuyết sai: []


## 8. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — câu hỏi duy nhất:** `n` có phải bội của 4 không?

```python
n % 4 == 0      # True nghĩa là bạn thua
```

**Đoạn 2 — đảo lại cho đúng yêu cầu "bạn có thắng không":**

```python
return n % 4 != 0
```

Ghép lại chỉ còn một dòng. Ngắn, nhưng phía sau là cả lập luận W/L ở trên.

In [11]:
def can_win_nim(n: int) -> bool:
    return n % 4 != 0

for n in [1, 2, 3, 4, 5, 7, 8, 12, 2**31 - 1]:
    print(n, "->", can_win_nim(n))

1 -> True
2 -> True
3 -> True
4 -> False
5 -> True
7 -> True
8 -> False
12 -> False
2147483647 -> True


### 8.1. Chiến lược cụ thể: nên lấy bao nhiêu viên?

Công thức chỉ cho biết thắng/thua. Để **thực sự chơi**, ta cần nước đi:
- Ở thế thắng: lấy `n % 4` viên (để lại bội của 4).
- Ở thế thua: không có nước nào cứu; lấy 1 viên và hy vọng đối thủ đi sai.

In [12]:
def best_move(n: int) -> int:
    r = n % 4
    return r if r != 0 else 1   # r = 0: thế thua, lấy tạm 1 viên

for n in [5, 6, 7, 8, 13]:
    print(f"n = {n:>2}: nên lấy {best_move(n)} -> còn {n - best_move(n)} viên cho đối thủ")

n =  5: nên lấy 1 -> còn 4 viên cho đối thủ
n =  6: nên lấy 2 -> còn 4 viên cho đối thủ
n =  7: nên lấy 3 -> còn 4 viên cho đối thủ
n =  8: nên lấy 1 -> còn 7 viên cho đối thủ
n = 13: nên lấy 1 -> còn 12 viên cho đối thủ


## 9. TRACE từng bước — mô phỏng một ván đấu hai người chơi tối ưu

Hàm `trace_game(n)` cho **cả hai người** dùng `best_move`. Ta in: lượt của ai, số đá trước, số lấy, số đá sau, và đối thủ có đứng ở bội của 4 không.

Bảng tay cho `n = 10`:

| Lượt | Người | Trước | Lấy | Sau | Sau có chia hết 4? |
|---:|---|---:|---:|---:|:---:|
| 1 | Bạn | 10 | 2 | 8 | có |
| 2 | Bạn kia | 8 | 1 | 7 | không |
| 3 | Bạn | 7 | 3 | 4 | có |
| 4 | Bạn kia | 4 | 1 | 3 | không |
| 5 | Bạn | 3 | 3 | 0 | có → bạn lấy viên cuối, **bạn thắng** |

In [13]:
def trace_game(n: int) -> str:
    players = ["Bạn", "Bạn kia"]
    turn = 0
    step = 1
    while True:
        who = players[turn]
        take = best_move(n)
        after = n - take
        print(f"Lượt {step}: {who:<7} | trước = {n:>2} | lấy {take} | sau = {after:>2} | sau % 4 == 0? {after % 4 == 0}")
        n = after
        if n == 0:
            print(f"=> {who} lấy viên cuối và THẮNG")
            return who
        turn = 1 - turn
        step += 1

trace_game(10)
print()
trace_game(8)   # bạn đi trước ở thế thua

Lượt 1: Bạn     | trước = 10 | lấy 2 | sau =  8 | sau % 4 == 0? True
Lượt 2: Bạn kia | trước =  8 | lấy 1 | sau =  7 | sau % 4 == 0? False
Lượt 3: Bạn     | trước =  7 | lấy 3 | sau =  4 | sau % 4 == 0? True
Lượt 4: Bạn kia | trước =  4 | lấy 1 | sau =  3 | sau % 4 == 0? False
Lượt 5: Bạn     | trước =  3 | lấy 3 | sau =  0 | sau % 4 == 0? True
=> Bạn lấy viên cuối và THẮNG

Lượt 1: Bạn     | trước =  8 | lấy 1 | sau =  7 | sau % 4 == 0? False
Lượt 2: Bạn kia | trước =  7 | lấy 3 | sau =  4 | sau % 4 == 0? True
Lượt 3: Bạn     | trước =  4 | lấy 1 | sau =  3 | sau % 4 == 0? False
Lượt 4: Bạn kia | trước =  3 | lấy 3 | sau =  0 | sau % 4 == 0? True
=> Bạn kia lấy viên cuối và THẮNG


'Bạn kia'

### 9.1. Trace bảng DP — xem từng ô được tính như thế nào

Ở đây ta in, với mỗi `i`, ba trạng thái đối thủ `i-1, i-2, i-3` và lý do `dp[i]` là W hay L.

In [14]:
def trace_dp(n: int) -> None:
    dp = [False] * (n + 1)
    print("i | đối thủ có thể đứng ở           | dp[i]")
    for i in range(1, n + 1):
        options = [(i - k, dp[i - k]) for k in (1, 2, 3) if i - k >= 0]
        dp[i] = any(not w for _, w in options)
        desc = ", ".join(f"{j}:{'W' if w else 'L'}" for j, w in options)
        print(f"{i:>1} | {desc:<32} | {'W' if dp[i] else 'L'}")

trace_dp(9)

i | đối thủ có thể đứng ở           | dp[i]
1 | 0:L                              | W
2 | 1:W, 0:L                         | W
3 | 2:W, 1:W, 0:L                    | W
4 | 3:W, 2:W, 1:W                    | L
5 | 4:L, 3:W, 2:W                    | W
6 | 5:W, 4:L, 3:W                    | W
7 | 6:W, 5:W, 4:L                    | W
8 | 7:W, 6:W, 5:W                    | L
9 | 8:L, 7:W, 6:W                    | W


## 10. Code nộp LeetCode

### Cách chính — công thức O(1)

In [15]:
class Solution:
    def canWinNim(self, n: int) -> bool:
        return n % 4 != 0

sol = Solution()
print(sol.canWinNim(4), sol.canWinNim(1), sol.canWinNim(2))   # False True True

False True True


### Cách tham khảo — DP với 3 biến (không nộp được với n lớn)

Chỉ cần nhớ 3 giá trị gần nhất nên bộ nhớ O(1), nhưng thời gian vẫn O(n) → với `n = 2^31 - 1` sẽ **Time Limit Exceeded**. Ta giữ để hiểu đường đi từ DP tới công thức.

In [16]:
class SolutionDP:
    def canWinNim(self, n: int) -> bool:
        a, b, c = False, True, True     # dp[0], dp[1], dp[2]
        if n <= 2:
            return [a, b, c][n]
        for _ in range(3, n + 1):
            d = (not a) or (not b) or (not c)   # dp[i] dựa vào dp[i-3], dp[i-2], dp[i-1]
            a, b, c = b, c, d
        return c

print([SolutionDP().canWinNim(n) for n in range(1, 13)])

[True, True, True, False, True, True, True, False, True, True, True, False]


## 11. Vì sao đúng? — chứng minh bằng invariant

**Mệnh đề:** Người đến lượt với `n` viên thua (khi đối thủ chơi tối ưu) ⇔ `n % 4 == 0`.

Chứng minh bằng quy nạp theo `n`:
- **Cơ sở:** `n = 0` là thế thua (đối thủ đã lấy viên cuối). `0 % 4 == 0` ✔.
- **Nếu `n % 4 == 0`, `n > 0`:** mọi nước lấy `k ∈ {1,2,3}` để lại `n - k` có `(n - k) % 4 ∈ {3, 2, 1}` ≠ 0 → theo giả thiết quy nạp, đối thủ đứng ở thế **thắng**. Không có nước nào tốt → thế **thua** ✔.
- **Nếu `n % 4 = r ≠ 0`:** lấy `k = r` (hợp lệ vì `1 <= r <= 3` và `r <= n`) để lại `n - r` chia hết cho 4 → đối thủ ở thế **thua** → mình ở thế **thắng** ✔.

**Invariant của chiến lược thắng:** "Mỗi khi đến lượt đối thủ, số đá là bội của 4." Bội của 4 nhỏ nhất là 0, và đối thủ đứng ở 0 nghĩa là mình vừa lấy viên cuối.

## 12. Time / Space Complexity và so sánh các cách

Gọi `n` là số viên đá.

| Cách | Ý tưởng | Time | Space | Nộp được? |
|---|---|---|---|:---:|
| Đệ quy thuần | thử mọi nước đi | `O(3^n)` (chặn trên), thực tế ~`O(1.84^n)` | `O(n)` stack | Không |
| Memo `lru_cache` | nhớ kết quả từng `n` | `O(n)` | `O(n)` + đệ quy sâu | Không (n lớn) |
| DP bảng | tính `dp[0..n]` | `O(n)` | `O(n)` | Không (n lớn) |
| DP 3 biến | chỉ nhớ 3 ô cuối | `O(n)` | `O(1)` | Không (TLE) |
| **Công thức** | `n % 4 != 0` | **`O(1)`** | **`O(1)`** | **Có** |

Bài học: DP/brute force không bị bỏ đi — nó là công cụ **phát hiện** và **kiểm chứng** quy luật.

## 13. Bộ test — mẫu đề, biên, đối chiếu nhiều cách

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
# Ví dụ trong đề
check("n=4", sol.canWinNim(4), False)
check("n=1", sol.canWinNim(1), True)
check("n=2", sol.canWinNim(2), True)
# Biên và trường hợp dễ sai
check("n=3", sol.canWinNim(3), True)
check("n=5 (gợi ý của đề)", sol.canWinNim(5), True)
check("n=8", sol.canWinNim(8), False)
check("n lớn nhất 2^31-1", sol.canWinNim(2**31 - 1), True)    # 2^31-1 = 4k + 3
check("n=2^31-4 (bội của 4)", sol.canWinNim(2**31 - 4), False)

PASS n=4: False
PASS n=1: True
PASS n=2: True
PASS n=3: True
PASS n=5 (gợi ý của đề): True
PASS n=8: False
PASS n lớn nhất 2^31-1: True
PASS n=2^31-4 (bội của 4): False


In [18]:
# Đối chiếu: công thức vs memo vs DP bảng vs DP 3 biến trên n = 1..300
dp = nim_table(300)
for n in range(1, 301):
    expected = dp[n]
    assert Solution().canWinNim(n) == expected, n
    assert SolutionDP().canWinNim(n) == expected, n
    assert can_win_memo(n) == expected, n
for n in range(1, 18):
    assert can_win_recursive(n) == dp[n], n
print("Tất cả cách cho cùng kết quả trên n = 1..300 (đệ quy thuần kiểm tới 17)")

Tất cả cách cho cùng kết quả trên n = 1..300 (đệ quy thuần kiểm tới 17)


## 14. Lỗi người mới hay gặp

### Lỗi 1: Đảo ngược điều kiện

```python
return n % 4 == 0      # SAI: đây là điều kiện THUA
```
Sửa: `return n % 4 != 0`.

### Lỗi 2: Dùng DP/đệ quy cho n tới 2^31 - 1
Đúng logic nhưng **TLE** (quá thời gian) hoặc **MemoryError** (`[False] * (n+1)` với 2 tỉ phần tử) hoặc **RecursionError**.

### Lỗi 3: Sai trường hợp cơ sở `win(0) = True`
Nghĩ "còn 0 viên là mình thắng" — nhưng đến lượt mình mà còn 0 viên nghĩa là **đối thủ** vừa lấy viên cuối. Hệ quả: toàn bộ bảng bị đảo.

### Lỗi 4: Quên điều kiện `k <= n` trong đệ quy
Gọi `win(n - 3)` khi `n = 1` → `win(-2)` → đệ quy không có điểm dừng.

In [19]:
def can_win_wrong_base(n: int) -> bool:
    dp = [True] * (n + 1)          # SAI: dp[0] = True
    for i in range(1, n + 1):
        dp[i] = any(not dp[i - k] for k in (1, 2, 3) if i - k >= 0)
    return dp[n]

for n in [1, 2, 4, 5]:
    print(n, "sai:", can_win_wrong_base(n), "| đúng:", n % 4 != 0)

1 sai: False | đúng: True
2 sai: True | đúng: True
4 sai: True | đúng: False
5 sai: False | đúng: True


In [20]:
def can_win_no_guard(n: int, depth: int = 0) -> bool:
    if depth > 20:                 # chặn để demo không chạy vô hạn
        raise RecursionError("đệ quy xuống số âm, không có điểm dừng")
    if n == 0:
        return False
    return any([not can_win_no_guard(n - k, depth + 1) for k in (1, 2, 3)])   # SAI: thiếu k <= n

try:
    can_win_no_guard(1)
except RecursionError as e:
    print("Lỗi:", e)

Lỗi: đệ quy xuống số âm, không có điểm dừng


## 15. Mẹo nhận dạng pattern

- Đề nói **"hai người chơi tối ưu"**, **"bạn đi trước"**, trả về thắng/thua → nghĩ ngay **thế W/L** và **DP trên trạng thái**.
- Nếu input lên tới `10^9` trở lên mà chỉ có **một số** → gần như chắc chắn có **công thức/quy luật tuần hoàn**.
- Quy trình chuẩn: **brute force/DP nhỏ → in bảng → đoán quy luật → chứng minh → kiểm chứng tự động**.
- Mỗi lượt lấy từ `1` tới `m` viên → chu kỳ là `m + 1` (bài này `m = 3` nên chu kỳ 4). Đây là **Bachet's game**.

## 16. Biến thể

### Biến thể 1 — mỗi lượt lấy từ 1 tới m viên
Người đi trước thua ⇔ `n % (m + 1) == 0`. Kiểm chứng bằng DP tổng quát.

In [21]:
def can_win_general_dp(n: int, moves) -> bool:
    dp = [False] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = any(not dp[i - k] for k in moves if i - k >= 0)
    return dp[n]

for m in [2, 3, 5]:
    ok = all(can_win_general_dp(n, range(1, m + 1)) == (n % (m + 1) != 0) for n in range(1, 60))
    print(f"m = {m}: công thức n % {m + 1} != 0 đúng trên n=1..59? {ok}")

m = 2: công thức n % 3 != 0 đúng trên n=1..59? True
m = 3: công thức n % 4 != 0 đúng trên n=1..59? True
m = 5: công thức n % 6 != 0 đúng trên n=1..59? True


### Biến thể 2 — tập nước đi bất kỳ (ví dụ chỉ lấy 1 hoặc 4 viên)
Không còn công thức đơn giản `% (m+1)`, nhưng DP vẫn tìm được và thường có **chu kỳ**. Với nước đi `{1, 4}`, thế thua lặp lại theo chu kỳ 5: `n % 5 ∈ {0, 2}`.

### Biến thể 3 — misère (ai lấy viên cuối thì THUA)
Thay điều kiện cơ sở: đến lượt mà còn 1 viên thì buộc lấy và thua. Với `m = 3`, người đi trước thua ⇔ `n % 4 == 1`.

In [22]:
dp_14 = [can_win_general_dp(n, (1, 4)) for n in range(0, 21)]
print("Thế thua với nước đi {1,4}:", [n for n in range(21) if not dp_14[n]])

def misere_dp(n: int) -> bool:
    # dp[i]: người đến lượt với i viên có thắng không, ai lấy viên cuối thì thua
    dp = [True] * (n + 1)          # dp[0]: đối thủ vừa lấy viên cuối -> đối thủ thua -> mình thắng
    for i in range(1, n + 1):
        dp[i] = any(not dp[i - k] for k in (1, 2, 3) if i - k >= 0)
    return dp[n]

print("Misère, thế thua:", [n for n in range(1, 21) if not misere_dp(n)])

Thế thua với nước đi {1,4}: [0, 2, 5, 7, 10, 12, 15, 17, 20]
Misère, thế thua: [1, 5, 9, 13, 17]


## 17. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
Lập bảng W/L cho `n = 0..10` khi mỗi lượt được lấy **1 hoặc 2** viên. Thế thua là những `n` nào?

### Bài 2 — Dự đoán output
Không chạy code, đoán: `[n for n in range(1, 21) if n % 4 == 0]` và `Solution().canWinNim(100)`.

### Bài 3 — Điền chỗ trống: bảng DP
Hoàn thiện `exercise_nim_dp(n)` trả về bảng `dp[0..n]`.

In [23]:
def exercise_nim_dp(n: int) -> list[bool]:
    dp = [False] * (n + 1)
    for i in range(1, n + 1):
        # TODO: dp[i] = True nếu có k trong (1,2,3), i-k >= 0, mà dp[i-k] là False
        pass
    return dp

# Tự làm xong rồi mới mở test:
# assert exercise_nim_dp(8) == [False, True, True, True, False, True, True, True, False]

### Bài 4 — Tự viết: nước đi tốt nhất
Viết `exercise_best_move(n)` trả về số viên nên lấy để thắng, hoặc `0` nếu đang ở thế thua.

In [24]:
def exercise_best_move(n: int) -> int:
    # TODO
    pass

# assert exercise_best_move(5) == 1
# assert exercise_best_move(7) == 3
# assert exercise_best_move(8) == 0

### Bài 5 — Biến thể: lấy từ 1 tới m viên
Viết `exercise_can_win_m(n, m)` chạy O(1).

In [25]:
def exercise_can_win_m(n: int, m: int) -> bool:
    # TODO
    pass

# assert exercise_can_win_m(4, 3) == False
# assert exercise_can_win_m(7, 5) == True
# assert exercise_can_win_m(6, 2) == False

### Bài 6 — Mini challenge: tìm chu kỳ tự động
Cho một tập nước đi `moves` (ví dụ `(1, 3, 4)`), viết `exercise_find_period(moves, limit)` tính bảng W/L tới `limit` rồi trả về **chu kỳ nhỏ nhất `p`** sao cho `dp[i] == dp[i + p]` với mọi `i` từ `max(moves)` tới `limit - p`.

In [26]:
def exercise_find_period(moves, limit: int = 200) -> int:
    # TODO 1: tính dp[0..limit]
    # TODO 2: thử p = 1, 2, ... và kiểm tra dp[i] == dp[i+p]
    pass

# assert exercise_find_period((1, 2, 3)) == 4
# assert exercise_find_period((1, 3, 4)) == 7

## 18. Gợi ý 3 mức

**Bài 1** — (1) Bắt đầu từ `0 = L`. (2) `n` là W nếu `n-1` hoặc `n-2` là L. (3) Chu kỳ là `m + 1 = 3`.

**Bài 2** — (1) Bội của 4 trong 1..20. (2) `100 % 4`. (3) `100 = 4 × 25`.

**Bài 3** — (1) Đây là định nghĩa W/L. (2) Dùng `any(...)` với generator. (3) `dp[i] = any(not dp[i - k] for k in (1, 2, 3) if i - k >= 0)`.

**Bài 4** — (1) Muốn để lại bội của 4. (2) Phần dư `n % 4` chính là số cần lấy. (3) Nếu phần dư bằng 0 thì trả 0.

**Bài 5** — (1) Mỗi cặp lượt giảm `m + 1`. (2) Tổng quát hóa `% 4` thành `% (m + 1)`. (3) `return n % (m + 1) != 0`.

**Bài 6** — (1) Dùng lại `can_win_general_dp` nhưng giữ cả bảng. (2) Vòng `for p in range(1, limit)`. (3) Bỏ qua phần đầu `i < max(moves)` vì ở đó một số nước đi chưa hợp lệ.

## 19. Đáp án tham khảo

**Bài 1:** lấy 1 hoặc 2 viên → thế thua `0, 3, 6, 9` (bội của 3).

**Bài 2:** `[4, 8, 12, 16, 20]`; `canWinNim(100) == False` vì `100 % 4 == 0`.

In [27]:
def answer_nim_dp(n: int) -> list[bool]:
    dp = [False] * (n + 1)
    for i in range(1, n + 1):
        dp[i] = any(not dp[i - k] for k in (1, 2, 3) if i - k >= 0)
    return dp

def answer_best_move(n: int) -> int:
    return n % 4

def answer_can_win_m(n: int, m: int) -> bool:
    return n % (m + 1) != 0

def answer_find_period(moves, limit: int = 200) -> int:
    dp = [False] * (limit + 1)
    for i in range(1, limit + 1):
        dp[i] = any(not dp[i - k] for k in moves if i - k >= 0)
    start = max(moves)
    for p in range(1, limit - start):
        if all(dp[i] == dp[i + p] for i in range(start, limit - p + 1)):
            return p
    return -1

# Bài 1 kiểm tra bằng DP tổng quát
assert [n for n in range(11) if not can_win_general_dp(n, (1, 2))] == [0, 3, 6, 9]
# Bài 2
assert [n for n in range(1, 21) if n % 4 == 0] == [4, 8, 12, 16, 20]
assert Solution().canWinNim(100) is False
# Bài 3..6
assert answer_nim_dp(8) == [False, True, True, True, False, True, True, True, False]
assert answer_best_move(5) == 1 and answer_best_move(7) == 3 and answer_best_move(8) == 0
assert answer_can_win_m(4, 3) is False and answer_can_win_m(7, 5) is True and answer_can_win_m(6, 2) is False
assert answer_find_period((1, 2, 3)) == 4
assert answer_find_period((1, 3, 4)) == 7
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 20. Template nhớ nhanh + checklist submit

```python
class Solution:
    def canWinNim(self, n: int) -> bool:
        return n % 4 != 0
```

**Template tư duy bài trò chơi:**
1. Định nghĩa trạng thái và thế thua cơ sở (`win(0) = False`).
2. `win(s) = any(not win(s') for s' in next_states(s))`.
3. Viết DP nhỏ, in bảng, tìm quy luật/chu kỳ.
4. Chứng minh quy luật (quy nạp hoặc chiến lược "đáp đối xứng").
5. So sánh công thức với DP trên nhiều `n` bằng `assert`.

**Checklist trước khi submit:**
- [ ] Trả về `True` khi **thắng** (không đảo điều kiện)?
- [ ] Không dùng vòng lặp/đệ quy phụ thuộc `n` (n tới 2^31 - 1)?
- [ ] Đã thử `n = 1, 2, 3, 4, 5, 8`?
- [ ] Chữ ký đúng `canWinNim(self, n: int) -> bool`?